# 02. 모델 학습 + 어블레이션

**런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 켤 것.

이전 학습 대비 바뀐 점
- 원본을 **시간순 정렬**한다 (이전에는 정렬이 없어 시퀀스가 역순이었고, 미래
  투구의 카운트로 정답을 읽을 수 있었다)
- 시퀀스를 **타석 안으로 제한**하고 패딩 마스크를 쓴다
- 분할이 **시즌 단위**다 (2024 학습 / 2025 테스트)
- 인코더·스케일러를 **학습 구간에만 fit** 한다
- `WeightedRandomSampler` 를 쓰지 않는다 (Focal Loss 와 이중 보정되어
  확률 캘리브레이션이 망가졌었다)
- 조기 종료 기준이 **검증 log-loss** 다 (정확도 노이즈 피크를 성능으로
  보고하던 문제를 막는다)

## 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# src/ 코드를 코랩으로 옮긴다.
#
# [방법 A] 저장소를 이미 푸시했다면 클론이 가장 간단하다.
#   !git clone -q https://github.com/hwangiljun/deeplearning_project.git /content/repo
#
# [방법 B] 아직 푸시 전이면 로컬에서 만든 zip 을 올린다.
#   로컬에서 먼저:  python scripts/make_colab_bundle.py
#   생성된 colab_src.zip 을 아래 업로드 창에서 선택.

import pathlib, zipfile

REPO = pathlib.Path('/content/repo')

if not (REPO / 'src').exists():
    from google.colab import files
    uploaded = files.upload()          # colab_src.zip 선택
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall(REPO)

%cd /content/repo
!pip install -q scikit-learn joblib
!ls src

In [ ]:
import gc, time, json

# 어블레이션은 '공정 비교'가 목적이므로 모든 구성에 **같은 예산**을 준다.
# 본 학습(40에폭)보다 짧게 잡아도 비교는 성립한다. 7개 구성 × 40에폭은
# 몇 시간이 걸리므로 15에폭 + patience 4 로 둔다.
ab_cfg = TrainConfig(epochs=15, batch_size=512, lr=3e-4, patience=4)

grid = ablation_grid(enc.cat_sizes)
results = {}
cache = {6: (train_data, test_data)}      # seq_len=6 은 이미 만들어 둔 것을 재사용

for i, (name, cfg) in enumerate(grid.items(), 1):
    t0 = time.time()
    print(f"
===== [{i}/{len(grid)}] {name} (seq_len={cfg.seq_len}) =====", flush=True)

    if cfg.seq_len not in cache:
        print(f"  seq_len={cfg.seq_len} 전처리 중...", flush=True)
        tr_i, te_i, _ = prepare(paths, train_seasons=[2024], test_seasons=[2025],
                                seq_len=cfg.seq_len)
        cache[cfg.seq_len] = (tr_i, te_i)
    tr_i, te_i = cache[cfg.seq_len]

    m, h, bv = train(tr_i, cfg, ab_cfg, verbose=False)
    loader = DataLoader(to_tensors(te_i), batch_size=2048)
    tm, _, _ = evaluate(m, loader, "cuda", len(OUTCOMES))

    results[name] = {"val": bv, "test": tm, "epochs": len(h)}
    print(f"  -> 테스트 acc {tm['accuracy']*100:.2f}% · macroF1 {tm['macro_f1']:.4f} "
          f"· AUC {tm['macro_auc']:.4f} · logloss {tm['log_loss']:.4f}  "
          f"[{(time.time()-t0)/60:.1f}분]", flush=True)

    # 중간에 끊겨도 지금까지의 결과는 남긴다
    (OUT / "ablation.json").write_text(json.dumps(results, indent=2, default=float),
                                       encoding="utf-8")

    # seq_len=6 외의 캐시는 메모리를 많이 먹으므로 바로 비운다
    if cfg.seq_len != 6:
        del cache[cfg.seq_len]
    del m
    gc.collect()
    torch.cuda.empty_cache()

print("
어블레이션 완료")

## 전처리

시즌 단위 분할이라 겹치는 윈도우가 학습/테스트로 갈리지 않는다.
타자 맥락 피처는 **직전 시즌** 성적만 쓴다 (2024 성적 → 2025 경기).

In [ ]:
paths = sorted(RAW.glob('*.parquet'))

train_data, test_data, enc = prepare(
    paths,
    train_seasons=[2024],
    test_seasons=[2025],
    seq_len=6,
)

print(f'학습 {len(train_data):,}구  /  테스트 {len(test_data):,}구')
print('범주 사전 크기:', enc.cat_sizes)
print('클래스 분포 (학습):')
for i, name in enumerate(OUTCOMES):
    print(f'  {name:18s} {(train_data.y == i).sum():8,}  {100*(train_data.y==i).mean():5.2f}%')

## 본 모델 학습

In [ ]:
model_cfg = ModelConfig(
    cat_sizes=enc.cat_sizes,
    d_model=128,
    nhead=4,
    num_layers=2,        # 논문 하이퍼파라미터 튜닝에서도 2개가 최적이었다
    dim_feedforward=512,
    seq_len=6,
    n_classes=len(OUTCOMES),
)
train_cfg = TrainConfig(epochs=40, batch_size=512, lr=3e-4, patience=6)

model, history, best_val = train(train_data, model_cfg, train_cfg)
print('\n최고 검증 지표:', json.dumps(best_val, indent=2, default=float))

## 테스트 시즌(2025) 평가

검증셋은 조기 종료·모델 선택에 이미 썼으므로, 성능은 **한 번도 건드리지 않은
2025 시즌**에서 잰다. 이전 보고서는 검증셋 최고 에폭 값을 그대로 성능으로
보고해서 낙관 편향이 있었다.

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(to_tensors(test_data), batch_size=1024)
test_metrics, probs, targets = evaluate(model, test_loader, 'cuda', len(OUTCOMES))

print('=== 2025 시즌 홀드아웃 ===')
for k, v in test_metrics.items():
    print(f'  {k:12s} {v:.4f}')

print('\n주변 확률 일치도 (캘리브레이션의 1차 점검):')
for i, name in enumerate(OUTCOMES):
    print(f'  {name:18s} 예측 {probs[:,i].mean():.4f}  실제 {(targets==i).mean():.4f}')

In [ ]:
save_artifacts(
    OUT / 'main',
    model, model_cfg, enc,
    history=history,
    metrics=best_val,
    extra={'test_metrics': test_metrics,
           'train_seasons': [2024], 'test_seasons': [2025]},
)
print('저장 완료:', OUT / 'main')

## 어블레이션

이전 보고서의 가장 큰 약점은 **어블레이션이 없다**는 것이었다. 제안 모델과
베이스라인이 d_model·레이어 수·손실함수·에폭·학습률까지 전부 달라서, 성능
차이를 어느 요소에 귀속시킬 수 없었다.

여기서는 **한 번에 하나씩만** 바꾼다. 나머지 조건은 전부 동일하다.

| 구성 | 검증하는 것 |
|---|---|
| `full` | 제안 모델 전체 |
| `no_context_skip` | 맥락 스킵 연결의 기여 (보고서의 핵심 주장) |
| `no_state_embed` | 카운트/베이스아웃 상태 임베딩의 기여 |
| `last_token` | 마스크드 평균 풀링 vs 마지막 토큰 (이전 방식) |
| `seq_len_1/3/10` | 시퀀스 길이의 기여 = 투구 배합이 실제로 정보를 주는가 |

`seq_len_1` 이 특히 중요하다. 시퀀스 길이 1 이면 직전 투구 정보가 전혀 없으므로,
`full` 이 `seq_len_1` 을 유의미하게 이기지 못하면 **"시계열 맥락이 중요하다"는
주장 자체가 성립하지 않는다.**

In [ ]:
results = {}
for name, cfg in ablation_grid(enc.cat_sizes).items():
    print(f'\n===== {name} =====')
    if cfg.seq_len != 6:
        # 시퀀스 길이가 다르면 데이터부터 다시 만든다
        tr_i, te_i, _ = prepare(paths, train_seasons=[2024], test_seasons=[2025],
                                seq_len=cfg.seq_len)
    else:
        tr_i, te_i = train_data, test_data

    m, h, bv = train(tr_i, cfg, train_cfg, verbose=False)
    loader = DataLoader(to_tensors(te_i), batch_size=1024)
    tm, _, _ = evaluate(m, loader, 'cuda', len(OUTCOMES))
    results[name] = {'val': bv, 'test': tm, 'epochs': len(h)}
    print(f"  검증 logloss {bv['log_loss']:.4f} | 테스트 acc {tm['accuracy']*100:.2f}% "
          f"macroF1 {tm['macro_f1']:.4f} AUC {tm['macro_auc']:.4f} logloss {tm['log_loss']:.4f}")

(OUT / 'ablation.json').write_text(json.dumps(results, indent=2, default=float),
                                   encoding='utf-8')

In [ ]:
import pandas as pd

rows = [{
    '구성': k,
    '정확도(%)': round(v['test']['accuracy'] * 100, 2),
    'macro-F1': round(v['test']['macro_f1'], 4),
    'macro-AUC': round(v['test']['macro_auc'], 4),
    'log-loss': round(v['test']['log_loss'], 4),
    '에폭': v['epochs'],
} for k, v in results.items()]

table = pd.DataFrame(rows).sort_values('log-loss')
display(table)
table.to_csv(OUT / 'ablation_table.csv', index=False, encoding='utf-8-sig')

## 산출물 내려받기

`artifacts/main/` 을 로컬 저장소의 `models/` 에 넣으면 앱이 바로 쓴다.

In [ ]:
import shutil
shutil.make_archive('/content/artifacts', 'zip', OUT)

from google.colab import files
files.download('/content/artifacts.zip')